In [1]:
import pandas as pd

# 1. Cargar dataset de Fábrega
ruta = r"C:\Users\Martín\Desktop\POLARIZACOIN\datos\ideologia\ideologia_congreso_chile_2002_2026_long_format.csv"
df_ideo = pd.read_csv(ruta)

# 2. Filtrar solo el Periodo_6 (2022-2026, electos en 2021)
df_p6 = df_ideo[df_ideo['Period'] == 'Periodo_6'].copy()

# 3. Eliminar diputados sin estimación ideológica
df_p6 = df_p6.dropna(subset=['wnom_ideology'])

print(f"Diputados en Periodo_6 con ideología válida: {len(df_p6)}")

# 4. Mapeo de partidos (party_1) a Bloques
mapeo_partidos = {
    # Chile Podemos +
    'UDI': 'CHILE PODEMOS +', 'RN': 'CHILE PODEMOS +', 'EVOP': 'CHILE PODEMOS +', 'PRI': 'CHILE PODEMOS +',
    
    # Nuevo Pacto Social
    'DC': 'NUEVO PACTO SOCIAL', 'PS': 'NUEVO PACTO SOCIAL', 'PPD': 'NUEVO PACTO SOCIAL',
    'PR': 'NUEVO PACTO SOCIAL', 'PL': 'NUEVO PACTO SOCIAL', 'LIBERAL': 'NUEVO PACTO SOCIAL', 'CIU': 'NUEVO PACTO SOCIAL',
    
    # Apruebo Dignidad
    'PC': 'APRUEBO DIGNIDAD', 'RD': 'APRUEBO DIGNIDAD', 'PCS': 'APRUEBO DIGNIDAD',
    'COMUNES': 'APRUEBO DIGNIDAD', 'FRVS': 'APRUEBO DIGNIDAD', 'AH': 'APRUEBO DIGNIDAD',
    
    # Frente Social Cristiano
    'PREP': 'FRENTE SOCIAL CRISTIANO', 'PSC': 'FRENTE SOCIAL CRISTIANO', 'PCC': 'FRENTE SOCIAL CRISTIANO',
    
    # Dignidad Ahora
    'PH': 'DIGNIDAD AHORA', 'IGUAL': 'DIGNIDAD AHORA',
    
    # Partidos fuera de pacto / Listas propias
    'PDG': 'PARTIDO DE LA GENTE', 
    'PEV': 'PARTIDO ECOLOGISTA VERDE',
    'PRO': 'PARTIDO PROGRESISTA DE CHILE', 
    'UP': 'UNION PATRIOTICA', 
    'PTR': 'PARTIDO DE TRABAJADORES REVOLUCIONARIOS',
    
    # Independientes
    'IND': 'CANDIDATURA INDEPENDIENTE'
}

# 5. Aplicar el mapeo
df_p6['Bloque'] = df_p6['party_1'].map(mapeo_partidos)

# Revisar si quedó algún partido sin mapear (que quedaría como NaN)
sin_mapear = df_p6[df_p6['Bloque'].isnull()]['party_1'].unique()
if len(sin_mapear) > 0:
    print(f"\nAtención: Partidos sin mapear: {sin_mapear}")
else:
    print("\nTodos los partidos fueron mapeados correctamente.")

# 6. Calcular la ideología promedio por bloque
resumen_bloques = df_p6.groupby('Bloque')['wnom_ideology'].agg(['count', 'mean', 'min', 'max']).round(3)
resumen_bloques = resumen_bloques.sort_values('mean')

print("\n=== POSICIÓN IDEOLÓGICA POR BLOQUE (Periodo 2022-2026) ===")
print("(Menor a 0 = Izquierda, Mayor a 0 = Derecha)\n")
print(resumen_bloques.to_string())

Diputados en Periodo_6 con ideología válida: 157

Todos los partidos fueron mapeados correctamente.

=== POSICIÓN IDEOLÓGICA POR BLOQUE (Periodo 2022-2026) ===
(Menor a 0 = Izquierda, Mayor a 0 = Derecha)

                           count   mean    min    max
Bloque                                               
APRUEBO DIGNIDAD              21 -0.914 -1.000 -0.790
DIGNIDAD AHORA                 3 -0.769 -0.963 -0.438
PARTIDO ECOLOGISTA VERDE       2 -0.731 -0.741 -0.720
NUEVO PACTO SOCIAL            29 -0.594 -0.761 -0.153
CANDIDATURA INDEPENDIENTE     37 -0.388 -0.983  0.996
PARTIDO DE LA GENTE            5  0.149 -0.180  0.369
CHILE PODEMOS +               49  0.526  0.028  0.950
FRENTE SOCIAL CRISTIANO       11  0.906  0.494  1.000


In [3]:
import pandas as pd
import glob
import os
import unicodedata
import re

def normalizar_nombre(nombre):
    if pd.isna(nombre):
        return None
    n = str(nombre).upper().strip()
    n = ''.join(c for c in unicodedata.normalize('NFD', n) if unicodedata.category(c) != 'Mn')
    n = re.sub(r'\s+', ' ', n).strip()
    return n

# 1. Leer los 16 archivos de SERVEL y extraer Candidato, Distrito, Comuna
archivos = sorted(glob.glob(r"C:\Users\Martín\Desktop\POLARIZACOIN\datos\Diputados\*.xlsx"))
lista_candidatos = []

print("Leyendo archivos SERVEL...")
for archivo in archivos:
    xls = pd.ExcelFile(archivo)
    for hoja in xls.sheet_names:
        df_temp = pd.read_excel(archivo, sheet_name=hoja, usecols=['Candidato', 'Distrito', 'Comuna'])
        lista_candidatos.append(df_temp)

df_serval_candidatos = pd.concat(lista_candidatos, ignore_index=True)

# Limpiar filas sin candidato (filas de totales o nulos)
df_serval_candidatos = df_serval_candidatos.dropna(subset=['Candidato'])
df_serval_candidatos = df_serval_candidatos.drop_duplicates().copy()

# Normalizar
df_serval_candidatos['nombre_norm'] = df_serval_candidatos['Candidato'].apply(normalizar_nombre)

print(f"Candidatos únicos en SERVEL: {len(df_serval_candidatos)}")

# 2. Suponiendo que df_p6 (Fábrega) sigue en memoria del paso anterior.
# Si no, tendrías que volver a cargarlo. Lo dejamos listo aquí por si acaso:
if 'df_p6' not in locals():
    ruta_ideo = r"C:\Users\Martín\Desktop\POLARIZACOIN\datos\ideologia\ideologia_congreso_chile_2002_2026_long_format.csv"
    df_ideo = pd.read_csv(ruta_ideo)
    df_p6 = df_ideo[df_ideo['Period'] == 'Periodo_6'].dropna(subset=['wnom_ideology']).copy()
    # Aplicar mapeo (asumiendo que tienes el diccionario mapeo_partidos del paso anterior)

# 3. Cruzar Fábrega con SERVEL
df_p6['nombre_norm'] = df_p6['Name'].apply(normalizar_nombre)

df_cruce = df_p6.merge(
    df_serval_candidatos[['nombre_norm', 'Distrito', 'Comuna']],
    on='nombre_norm',
    how='left'
)

# 4. Ver resultados
total = len(df_cruce)
cruzados = df_cruce['Distrito'].notna().sum()
print(f"\nTotal diputados Fábrega: {total}")
print(f"Diputados cruzados con distrito: {cruzados}")
print(f"Diputados sin distrito: {total - cruzados}")

if total - cruzados > 0:
    print("\n=== DIPUTADOS NO ENCONTRADOS EN SERVEL ===")
    print(df_cruce[df_cruce['Distrito'].isna()][['Name', 'party_1']].to_string())

Leyendo archivos SERVEL...
Candidatos únicos en SERVEL: 15909

Total diputados Fábrega: 1351
Diputados cruzados con distrito: 1300
Diputados sin distrito: 51

=== DIPUTADOS NO ENCONTRADOS EN SERVEL ===
                              Name  party_1
53            Gaspar Rivas Sánchez       RN
54           Frank Sauerbaum Muñoz       RN
55            Bernardo Berger Fett       RN
56             Karol Cariola Oliva       PC
141           Boris Barrera Moreno       PC
142   Alejandro Bernales Maldonado  LIBERAL
191           Sofía Cid Versalovic       RN
211           Camila Flores Oporto       RN
212           Juan Fuenzalida Cobo      UDI
224             Carmen Hertz Cádiz       PC
230          Diego Ibáñez Cotroneo      IND
346           Miguel Mellado Suazo       RN
347            Claudia Mix Jiménez  COMUNES
377     Erika Olivera De La Fuente      IND
446        Camila Rojas Valderrama    IGUAL
447           Leonidas Romero Sáez       RN
448      Patricio Rosas Barrientos      IND
472   

In [4]:
import pandas as pd
import unicodedata
import re

def normalizar_nombre(nombre):
    if pd.isna(nombre): return None
    n = str(nombre).upper().strip()
    n = ''.join(c for c in unicodedata.normalize('NFD', n) if unicodedata.category(c) != 'Mn')
    return re.sub(r'\s+', ' ', n).strip()

# 1. Volver a cargar Fábrega asegurando el Periodo_6
ruta_ideo = r"C:\Users\Martín\Desktop\POLARIZACOIN\datos\ideologia\ideologia_congreso_chile_2002_2026_long_format.csv"
df_ideo = pd.read_csv(ruta_ideo)
df_p6 = df_ideo[(df_ideo['Period'] == 'Periodo_6') & (df_ideo['wnom_ideology'].notna())].copy()

# Re-aplicar mapeo de bloques
mapeo_partidos = {
    'UDI': 'CHILE PODEMOS +', 'RN': 'CHILE PODEMOS +', 'EVOP': 'CHILE PODEMOS +', 'PRI': 'CHILE PODEMOS +',
    'DC': 'NUEVO PACTO SOCIAL', 'PS': 'NUEVO PACTO SOCIAL', 'PPD': 'NUEVO PACTO SOCIAL', 'PR': 'NUEVO PACTO SOCIAL', 'PL': 'NUEVO PACTO SOCIAL', 'LIBERAL': 'NUEVO PACTO SOCIAL', 'CIU': 'NUEVO PACTO SOCIAL',
    'PC': 'APRUEBO DIGNIDAD', 'RD': 'APRUEBO DIGNIDAD', 'PCS': 'APRUEBO DIGNIDAD', 'COMUNES': 'APRUEBO DIGNIDAD', 'FRVS': 'APRUEBO DIGNIDAD', 'AH': 'APRUEBO DIGNIDAD',
    'PREP': 'FRENTE SOCIAL CRISTIANO', 'PSC': 'FRENTE SOCIAL CRISTIANO', 'PCC': 'FRENTE SOCIAL CRISTIANO',
    'PH': 'DIGNIDAD AHORA', 'IGUAL': 'DIGNIDAD AHORA',
    'PDG': 'PARTIDO DE LA GENTE', 'PEV': 'PARTIDO ECOLOGISTA VERDE',
    'PRO': 'PARTIDO PROGRESISTA DE CHILE', 'UP': 'UNION PATRIOTICA', 'PTR': 'PARTIDO DE TRABAJADORES REVOLUCIONARIOS',
    'IND': 'CANDIDATURA INDEPENDIENTE'
}
df_p6['Bloque'] = df_p6['party_1'].map(mapeo_partidos)

# 2. Cargar SERVEL (asumiendo que df_serval_candidatos ya está en memoria del paso anterior)
if 'df_serval_candidatos' not in locals():
    print("Error: df_serval_candidatos no está en memoria. Vuelve a ejecutar la celda anterior.")
else:
    df_p6['nombre_norm'] = df_p6['Name'].apply(normalizar_nombre)

    # 3. Primer intento de cruce (nombre completo)
    df_cruce = df_p6.merge(
        df_serval_candidatos[['nombre_norm', 'Distrito', 'Comuna']],
        on='nombre_norm', how='left'
    )

    # 4. Plan de rescate para los que fallaron
    faltantes = df_cruce[df_cruce['Distrito'].isna()].copy()
    if len(faltantes) > 0:
        print(f"Rescatando {len(faltantes)} diputados que no cruzaron por nombre completo...")
        
        # Crear una versión corta: solo primer nombre y primer apellido
        df_serval_candidatos['nombre_corto'] = df_serval_candidatos['nombre_norm'].apply(lambda x: ' '.join(x.split()[:2]))
        df_cruce['nombre_corto'] = df_cruce['nombre_norm'].apply(lambda x: ' '.join(x.split()[:2]))

        # Cruce por nombre corto
        df_rescate = df_cruce[df_cruce['Distrito'].isna()].drop(columns=['Distrito', 'Comuna']).merge(
            df_serval_candidatos[['nombre_corto', 'Distrito', 'Comuna']].drop_duplicates('nombre_corto'),
            on='nombre_corto', how='left'
        )
        
        # Actualizar los que se rescataron
        df_cruce.update(df_rescate)
        
        # Eliminar la columna temporal
        df_cruce = df_cruce.drop(columns=['nombre_corto'])

    # Ver resultados finales
    total = len(df_cruce)
    cruzados = df_cruce['Distrito'].notna().sum()
    print(f"\nTotal diputados Periodo_6: {total}")
    print(f"Diputados cruzados con distrito: {cruzados}")
    print(f"Diputados sin distrito: {total - cruzados}")

    if total - cruzados > 0:
        print("\n=== DIPUTADOS QUE AÚN FALLAN ===")
        print(df_cruce[df_cruce['Distrito'].isna()][['Name', 'party_1']].to_string())

Rescatando 51 diputados que no cruzaron por nombre completo...

Total diputados Periodo_6: 1351
Diputados cruzados con distrito: 1300
Diputados sin distrito: 51

=== DIPUTADOS QUE AÚN FALLAN ===
                              Name  party_1
53            Gaspar Rivas Sánchez       RN
54           Frank Sauerbaum Muñoz       RN
55            Bernardo Berger Fett       RN
56             Karol Cariola Oliva       PC
141           Boris Barrera Moreno       PC
142   Alejandro Bernales Maldonado  LIBERAL
191           Sofía Cid Versalovic       RN
211           Camila Flores Oporto       RN
212           Juan Fuenzalida Cobo      UDI
224             Carmen Hertz Cádiz       PC
230          Diego Ibáñez Cotroneo      IND
346           Miguel Mellado Suazo       RN
347            Claudia Mix Jiménez  COMUNES
377     Erika Olivera De La Fuente      IND
446        Camila Rojas Valderrama    IGUAL
447           Leonidas Romero Sáez       RN
448      Patricio Rosas Barrientos      IND
472       Die

In [5]:
import pandas as pd
import unicodedata
import re

def normalizar_nombre(nombre):
    if pd.isna(nombre): return None
    n = str(nombre).upper().strip()
    n = ''.join(c for c in unicodedata.normalize('NFD', n) if unicodedata.category(c) != 'Mn')
    return re.sub(r'\s+', ' ', n).strip()

# 1. Volver a cargar Fábrega asegurando el Periodo_6 (157 diputados)
ruta_ideo = r"C:\Users\Martín\Desktop\POLARIZACOIN\datos\ideologia\ideologia_congreso_chile_2002_2026_long_format.csv"
df_ideo = pd.read_csv(ruta_ideo)
df_p6 = df_ideo[(df_ideo['Period'] == 'Periodo_6') & (df_ideo['wnom_ideology'].notna())].copy()

# Aplicar mapeo de bloques (por si acaso)
mapeo_partidos = {
    'UDI': 'CHILE PODEMOS +', 'RN': 'CHILE PODEMOS +', 'EVOP': 'CHILE PODEMOS +', 'PRI': 'CHILE PODEMOS +',
    'DC': 'NUEVO PACTO SOCIAL', 'PS': 'NUEVO PACTO SOCIAL', 'PPD': 'NUEVO PACTO SOCIAL', 'PR': 'NUEVO PACTO SOCIAL', 'PL': 'NUEVO PACTO SOCIAL', 'LIBERAL': 'NUEVO PACTO SOCIAL', 'CIU': 'NUEVO PACTO SOCIAL',
    'PC': 'APRUEBO DIGNIDAD', 'RD': 'APRUEBO DIGNIDAD', 'PCS': 'APRUEBO DIGNIDAD', 'COMUNES': 'APRUEBO DIGNIDAD', 'FRVS': 'APRUEBO DIGNIDAD', 'AH': 'APRUEBO DIGNIDAD',
    'PREP': 'FRENTE SOCIAL CRISTIANO', 'PSC': 'FRENTE SOCIAL CRISTIANO', 'PCC': 'FRENTE SOCIAL CRISTIANO',
    'PH': 'DIGNIDAD AHORA', 'IGUAL': 'DIGNIDAD AHORA',
    'PDG': 'PARTIDO DE LA GENTE', 'PEV': 'PARTIDO ECOLOGISTA VERDE',
    'PRO': 'PARTIDO PROGRESISTA DE CHILE', 'UP': 'UNION PATRIOTICA', 'PTR': 'PARTIDO DE TRABAJADORES REVOLUCIONARIOS',
    'IND': 'CANDIDATURA INDEPENDIENTE'
}
df_p6['Bloque'] = df_p6['party_1'].map(mapeo_partidos)
df_p6['nombre_norm'] = df_p6['Name'].apply(normalizar_nombre)

# 2. Asegurar que SERVEL tenga UN SOLO distrito por candidato (drop_duplicates por nombre)
if 'df_serval_candidatos' not in locals():
    print("Recargando SERVEL...")
    import glob
    archivos = sorted(glob.glob(r"C:\Users\Martín\Desktop\POLARIZACOIN\datos\Diputados\*.xlsx"))
    lista = []
    for a in archivos:
        xls = pd.ExcelFile(a)
        for h in xls.sheet_names:
            lista.append(pd.read_excel(a, sheet_name=h, usecols=['Candidato', 'Distrito', 'Comuna']))
    df_serval_candidatos = pd.concat(lista, ignore_index=True).dropna(subset=['Candidato'])
    df_serval_candidatos['nombre_norm'] = df_serval_candidatos['Candidato'].apply(normalizar_nombre)

# Limpiar SERVEL: un solo distrito por candidato
df_serval_unique = df_serval_candidatos.drop_duplicates(subset=['nombre_norm'], keep='first')

# 3. Cruce inicial
df_cruce = df_p6.merge(
    df_serval_unique[['nombre_norm', 'Distrito']],
    on='nombre_norm', how='left'
)

# 4. Plan de rescate por nombre corto (primer nombre y primer apellido)
faltantes_mask = df_cruce['Distrito'].isna()
if faltantes_mask.sum() > 0:
    print(f"Rescatando {faltantes_mask.sum()} diputados...")
    
    df_serval_unique['nombre_corto'] = df_serval_unique['nombre_norm'].apply(lambda x: ' '.join(x.split()[:2]))
    df_cruce['nombre_corto'] = df_cruce['nombre_norm'].apply(lambda x: ' '.join(x.split()[:2]))
    
    df_rescate = df_cruce[faltantes_mask].drop(columns=['Distrito']).merge(
        df_serval_unique[['nombre_corto', 'Distrito']].drop_duplicates('nombre_corto'),
        on='nombre_corto', how='left'
    )
    # Actualizar los distritos rescatados
    df_cruce.loc[faltantes_mask, 'Distrito'] = df_rescate['Distrito'].values
    
    df_cruce = df_cruce.drop(columns=['nombre_corto'])

# 5. Resultados
total = len(df_cruce)
cruzados = df_cruce['Distrito'].notna().sum()
print(f"\nTotal diputados Periodo_6: {total}")
print(f"Diputados cruzados con distrito: {cruzados}")
print(f"Diputados sin distrito: {total - cruzados}")

if total - cruzados > 0:
    print("\n=== DIPUTADOS QUE AÚN FALLAN ===")
    print(df_cruce[df_cruce['Distrito'].isna()][['Name', 'party_1']].to_string())

Rescatando 51 diputados...

Total diputados Periodo_6: 157
Diputados cruzados con distrito: 107
Diputados sin distrito: 50

=== DIPUTADOS QUE AÚN FALLAN ===
                             Name  party_1
5            Gaspar Rivas Sánchez       RN
6           Frank Sauerbaum Muñoz       RN
7            Bernardo Berger Fett       RN
8             Karol Cariola Oliva       PC
18           Boris Barrera Moreno       PC
19   Alejandro Bernales Maldonado  LIBERAL
25           Sofía Cid Versalovic       RN
29           Camila Flores Oporto       RN
30           Juan Fuenzalida Cobo      UDI
32             Carmen Hertz Cádiz       PC
34          Diego Ibáñez Cotroneo      IND
42           Miguel Mellado Suazo       RN
43            Claudia Mix Jiménez  COMUNES
47     Erika Olivera De La Fuente      IND
54        Camila Rojas Valderrama    IGUAL
55           Leonidas Romero Sáez       RN
56      Patricio Rosas Barrientos      IND
59       Diego Schalper Sepúlveda       RN
65             Gael Yeoman

In [6]:
import pandas as pd

# 1. Vamos a buscar a Karol Cariola, Gaspar Rivas y Claudia Mix en SERVEL
busquedas = ['CARIOLA', 'RIVAS', 'MIX']

for termino in busquedas:
    print(f"\n=== BUSCANDO '{termino}' EN SERVEL ===")
    # Filtrar donde el nombre contenga el término
    coincidencias = df_serval_candidatos[
        df_serval_candidatos['Candidato'].str.upper().str.contains(termino, na=False)
    ]['Candidato'].unique()
    
    print(f"Coincidencias encontradas: {len(coincidencias)}")
    for c in coincidencias:
        print(f"  - '{c}'")

# 2. Comparar con cómo están en Fábrega
print("\n=== CÓMO ESTÁN EN FÁBREGA ===")
nombres_fabrega = ['Karol Cariola Oliva', 'Gaspar Rivas Sánchez', 'Claudia Mix Jiménez']
print(df_p6[df_p6['Name'].isin(nombres_fabrega)]['Name'].tolist())


=== BUSCANDO 'CARIOLA' EN SERVEL ===
Coincidencias encontradas: 1
  - 'KAROL AIDA CARIOLA OLIVA'

=== BUSCANDO 'RIVAS' EN SERVEL ===
Coincidencias encontradas: 5
  - 'NATACHA ODETTE RIVAS MORALES'
  - 'GASPAR ALBERTO RIVAS SANCHEZ'
  - 'CLAUDIO FRANCISCO RIVAS CARRASCO'
  - 'CRISTIAN EDUARDO PUENTES RIVAS'
  - 'BRIAN RIVAS TIZNADO'

=== BUSCANDO 'MIX' EN SERVEL ===
Coincidencias encontradas: 1
  - 'CLAUDIA NATHALIE MIX JIMENEZ'

=== CÓMO ESTÁN EN FÁBREGA ===
['Gaspar Rivas Sánchez', 'Karol Cariola Oliva', 'Claudia Mix Jiménez']


In [7]:
import pandas as pd

# Función para extraer solo el primer nombre y el primer apellido
def extraer_nombre_apellido(nombre_completo):
    if pd.isna(nombre_completo): return None
    palabras = str(nombre_completo).split()
    if len(palabras) >= 2:
        return f"{palabras[0]} {palabras[-2]}" # Primera palabra + Penúltima palabra
    return palabras[0]

# 1. Aplicar la regla a SERVEL
df_serval_unique['clave_cruce'] = df_serval_unique['Candidato'].apply(extraer_nombre_apellido)

# 2. Aplicar la regla a Fábrega
df_p6['clave_cruce'] = df_p6['Name'].apply(extraer_nombre_apellido)

# 3. Volver a intentar el cruce inicial (por nombre normalizado, que ya funcionó para 107)
df_cruce = df_p6.merge(
    df_serval_unique[['nombre_norm', 'Distrito']],
    on='nombre_norm', how='left'
)

# 4. Plan de rescate para los que fallaron, usando la nueva clave_cruce
faltantes_mask = df_cruce['Distrito'].isna()
if faltantes_mask.sum() > 0:
    print(f"Rescatando {faltantes_mask.sum()} diputados con la nueva regla...")
    
    # Hacemos el merge solo con los que fallaron
    df_rescate = df_cruce[faltantes_mask].drop(columns=['Distrito']).merge(
        df_serval_unique[['clave_cruce', 'Distrito']].drop_duplicates('clave_cruce'),
        on='clave_cruce', how='left'
    )
    
    # Actualizar los distritos rescatados
    df_cruce.loc[faltantes_mask, 'Distrito'] = df_rescate['Distrito'].values

# 5. Resultados finales
total = len(df_cruce)
cruzados = df_cruce['Distrito'].notna().sum()
print(f"\nTotal diputados Periodo_6: {total}")
print(f"Diputados cruzados con distrito: {cruzados}")
print(f"Diputados sin distrito: {total - cruzados}")

if total - cruzados > 0:
    print("\n=== DIPUTADOS QUE AÚN FALLAN ===")
    print(df_cruce[df_cruce['Distrito'].isna()][['Name', 'party_1']].to_string())

Rescatando 51 diputados con la nueva regla...

Total diputados Periodo_6: 157
Diputados cruzados con distrito: 106
Diputados sin distrito: 51

=== DIPUTADOS QUE AÚN FALLAN ===
                             Name  party_1
5            Gaspar Rivas Sánchez       RN
6           Frank Sauerbaum Muñoz       RN
7            Bernardo Berger Fett       RN
8             Karol Cariola Oliva       PC
18           Boris Barrera Moreno       PC
19   Alejandro Bernales Maldonado  LIBERAL
25           Sofía Cid Versalovic       RN
29           Camila Flores Oporto       RN
30           Juan Fuenzalida Cobo      UDI
32             Carmen Hertz Cádiz       PC
34          Diego Ibáñez Cotroneo      IND
42           Miguel Mellado Suazo       RN
43            Claudia Mix Jiménez  COMUNES
47     Erika Olivera De La Fuente      IND
54        Camila Rojas Valderrama    IGUAL
55           Leonidas Romero Sáez       RN
56      Patricio Rosas Barrientos      IND
59       Diego Schalper Sepúlveda       RN
65     

In [8]:
import pandas as pd
import unicodedata

def crear_clave_corta(nombre):
    if pd.isna(nombre): return None
    # 1. Normalizar (mayúsculas y sin tildes)
    n = str(nombre).upper().strip()
    n = ''.join(c for c in unicodedata.normalize('NFD', n) if unicodedata.category(c) != 'Mn')
    # 2. Extraer primera palabra y penúltima palabra
    palabras = n.split()
    if len(palabras) >= 2:
        return f"{palabras[0]} {palabras[-2]}"
    return palabras[0]

# 1. Aplicar la regla a SERVEL
df_serval_unique['clave_cruce'] = df_serval_unique['Candidato'].apply(crear_clave_corta)

# 2. Aplicar la regla a Fábrega
df_p6['clave_cruce'] = df_p6['Name'].apply(crear_clave_corta)

# 3. Intentar rescatar a los que fallaron
faltantes_mask = df_cruce['Distrito'].isna()
print(f"Rescatando {faltantes_mask.sum()} diputados...")

if faltantes_mask.sum() > 0:
    # Hacemos el merge solo con los que fallaron
    df_rescate = df_cruce[faltantes_mask].drop(columns=['Distrito']).merge(
        df_serval_unique[['clave_cruce', 'Distrito']].drop_duplicates('clave_cruce'),
        on='clave_cruce', how='left'
    )
    
    # Actualizar los distritos rescatados
    df_cruce.loc[faltantes_mask, 'Distrito'] = df_rescate['Distrito'].values

# 4. Resultados finales
total = len(df_cruce)
cruzados = df_cruce['Distrito'].notna().sum()
print(f"\nTotal diputados Periodo_6: {total}")
print(f"Diputados cruzados con distrito: {cruzados}")
print(f"Diputados sin distrito: {total - cruzados}")

if total - cruzados > 0:
    print("\n=== DIPUTADOS QUE AÚN FALLAN ===")
    print(df_cruce[df_cruce['Distrito'].isna()][['Name', 'party_1']].to_string())

Rescatando 51 diputados...

Total diputados Periodo_6: 157
Diputados cruzados con distrito: 106
Diputados sin distrito: 51

=== DIPUTADOS QUE AÚN FALLAN ===
                             Name  party_1
5            Gaspar Rivas Sánchez       RN
6           Frank Sauerbaum Muñoz       RN
7            Bernardo Berger Fett       RN
8             Karol Cariola Oliva       PC
18           Boris Barrera Moreno       PC
19   Alejandro Bernales Maldonado  LIBERAL
25           Sofía Cid Versalovic       RN
29           Camila Flores Oporto       RN
30           Juan Fuenzalida Cobo      UDI
32             Carmen Hertz Cádiz       PC
34          Diego Ibáñez Cotroneo      IND
42           Miguel Mellado Suazo       RN
43            Claudia Mix Jiménez  COMUNES
47     Erika Olivera De La Fuente      IND
54        Camila Rojas Valderrama    IGUAL
55           Leonidas Romero Sáez       RN
56      Patricio Rosas Barrientos      IND
59       Diego Schalper Sepúlveda       RN
65             Gael Yeoman

In [9]:
import pandas as pd
import glob
import difflib
import unicodedata
import re

def normalizar_nombre(nombre):
    if pd.isna(nombre): return ""
    n = str(nombre).upper().strip()
    n = ''.join(c for c in unicodedata.normalize('NFD', n) if unicodedata.category(c) != 'Mn')
    return re.sub(r'\s+', ' ', n).strip()

# 1. Cargar Fábrega
ruta_ideo = r"C:\Users\Martín\Desktop\POLARIZACOIN\datos\ideologia\ideologia_congreso_chile_2002_2026_long_format.csv"
df_ideo = pd.read_csv(ruta_ideo)
df_p6 = df_ideo[(df_ideo['Period'] == 'Periodo_6') & (df_ideo['wnom_ideology'].notna())].copy()
df_p6['nombre_norm'] = df_p6['Name'].apply(normalizar_nombre)

# 2. Cargar SERVEL (solo los nombres y distritos)
archivos = sorted(glob.glob(r"C:\Users\Martín\Desktop\POLARIZACOIN\datos\Diputados\*.xlsx"))
lista = []
for a in archivos:
    xls = pd.ExcelFile(a)
    for h in xls.sheet_names:
        lista.append(pd.read_excel(a, sheet_name=h, usecols=['Candidato', 'Distrito']))
df_serval = pd.concat(lista, ignore_index=True).dropna(subset=['Candidato'])
df_serval['nombre_norm'] = df_serval['Candidato'].apply(normalizar_nombre)
df_serval_unique = df_serval.drop_duplicates(subset=['nombre_norm']).copy()

# 3. Magia con difflib
lista_serval = df_serval_unique['nombre_norm'].tolist()
distritos = []

print("Cruzando nombres con difflib (esto puede tardar unos segundos)...")

for nombre_fab in df_p6['nombre_norm']:
    # get_close_matches busca los strings más parecidos. n=1 pide solo el mejor. cutoff=0.8 exige 80% similitud.
    match = difflib.get_close_matches(nombre_fab, lista_serval, n=1, cutoff=0.8)
    if match:
        # Buscar el distrito en SERVEL para ese match
        distrito = df_serval_unique[df_serval_unique['nombre_norm'] == match[0]]['Distrito'].iloc[0]
        distritos.append(distrito)
    else:
        distritos.append(None)

df_p6['Distrito'] = distritos

# 4. Resultados
cruzados = df_p6['Distrito'].notna().sum()
print(f"\nTotal diputados: {len(df_p6)}")
print(f"Cruzados con distrito: {cruzados}")
print(f"Sin distrito: {len(df_p6) - cruzados}")

if len(df_p6) - cruzados > 0:
    print("\n=== DIPUTADOS QUE FALLARON (similitud < 80%) ===")
    print(df_p6[df_p6['Distrito'].isna()][['Name', 'party_1']].to_string())

Cruzando nombres con difflib (esto puede tardar unos segundos)...

Total diputados: 157
Cruzados con distrito: 151
Sin distrito: 6

=== DIPUTADOS QUE FALLARON (similitud < 80%) ===
                           Name party_1
2315   Daniel Manouchehri Lobos      PS
2465          Jaime Sáez Quiroz      RD
2483          Luis Sánchez Ossa    PREP
2543      Consuelo Veloso Ávila      RD
2567  Roberto Celedón Fernández     IND
2573      Arturo Barrios Oteíza      PS


In [10]:
import pandas as pd

# 1. Quedarnos solo con los diputados que tienen distrito y su bloque ya asignado
# (Asumimos que df_p6 tiene la columna 'Bloque' del paso anterior, si no, hay que re-mapearla)
mapeo_partidos = {
    'UDI': 'CHILE PODEMOS +', 'RN': 'CHILE PODEMOS +', 'EVOP': 'CHILE PODEMOS +', 'PRI': 'CHILE PODEMOS +',
    'DC': 'NUEVO PACTO SOCIAL', 'PS': 'NUEVO PACTO SOCIAL', 'PPD': 'NUEVO PACTO SOCIAL', 'PR': 'NUEVO PACTO SOCIAL', 'PL': 'NUEVO PACTO SOCIAL', 'LIBERAL': 'NUEVO PACTO SOCIAL', 'CIU': 'NUEVO PACTO SOCIAL',
    'PC': 'APRUEBO DIGNIDAD', 'RD': 'APRUEBO DIGNIDAD', 'PCS': 'APRUEBO DIGNIDAD', 'COMUNES': 'APRUEBO DIGNIDAD', 'FRVS': 'APRUEBO DIGNIDAD', 'AH': 'APRUEBO DIGNIDAD',
    'PREP': 'FRENTE SOCIAL CRISTIANO', 'PSC': 'FRENTE SOCIAL CRISTIANO', 'PCC': 'FRENTE SOCIAL CRISTIANO',
    'PH': 'DIGNIDAD AHORA', 'IGUAL': 'DIGNIDAD AHORA',
    'PDG': 'PARTIDO DE LA GENTE', 'PEV': 'PARTIDO ECOLOGISTA VERDE',
    'PRO': 'PARTIDO PROGRESISTA DE CHILE', 'UP': 'UNION PATRIOTICA', 'PTR': 'PARTIDO DE TRABAJADORES REVOLUCIONARIOS',
    'IND': 'CANDIDATURA INDEPENDIENTE'
}

if 'Bloque' not in df_p6.columns:
    df_p6['Bloque'] = df_p6['party_1'].map(mapeo_partidos)

df_con_distrito = df_p6.dropna(subset=['Distrito']).copy()
df_con_distrito['Distrito'] = df_con_distrito['Distrito'].astype(int)

# 2. Calcular la posición promedio de cada bloque EN CADA DISTRITO
# (Algunos distritos pueden no tener diputados de ciertos bloques)
posicion_por_distrito = df_con_distrito.groupby(['Distrito', 'Bloque'])['wnom_ideology'].mean().reset_index()

# 3. Calcular los extremos por distrito
extremos = posicion_por_distrito.groupby('Distrito')['wnom_ideology'].agg(['min', 'max']).reset_index()
extremos = extremos.rename(columns={'min': 'ideologia_min', 'max': 'ideologia_max'})

# Calcular el índice de polarización (Distancia entre extremos)
extremos['polarizacion_distrito'] = extremos['ideologia_max'] - extremos['ideologia_min']

# 4. Mostrar resultados
print("=== POLARIZACIÓN POR DISTRITO (2022-2026) ===")
print(f"Distritos calculados: {len(extremos)}")
print(extremos.sort_values('polarizacion_distrito', ascending=False).head(10).to_string(index=False))

# 5. Guardar el resultado para el cruce final
extremos.to_csv(r"C:\Users\Martín\Desktop\POLARIZACOIN\resultados\polarizacion_por_distrito.csv", index=False, encoding='utf-8-sig')
print("\nGuardado: resultados/polarizacion_por_distrito.csv")

=== POLARIZACIÓN POR DISTRITO (2022-2026) ===
Distritos calculados: 28
 Distrito  ideologia_min  ideologia_max  polarizacion_distrito
        8      -0.946934       0.980550               1.927484
       10      -0.927363       1.000000               1.927363
       11      -0.962690       0.941208               1.903898
        6      -0.925891       0.958119               1.884011
        9      -0.910824       0.966965               1.877790
        2      -1.000000       0.749249               1.749249
       14      -0.783822       0.958865               1.742687
       21      -0.875600       0.755226               1.630826
       20      -0.959760       0.663315               1.623075
       23      -0.900730       0.586802               1.487532

Guardado: resultados/polarizacion_por_distrito.csv


In [11]:
import pandas as pd

# 1. Cargar datasets
ruta_completo = r"C:\Users\Martín\Desktop\POLARIZACOIN\resultados\dataset_comuna_completo.csv"
ruta_pol = r"C:\Users\Martín\Desktop\POLARIZACOIN\resultados\polarizacion_por_distrito.csv"

df_comuna = pd.read_csv(ruta_completo)
df_pol = pd.read_csv(ruta_pol)

# 2. Asegurar que 'Distrito' sea del mismo tipo en ambos (int)
df_comuna['Distrito'] = df_comuna['Distrito'].astype(int)
df_pol['Distrito'] = df_pol['Distrito'].astype(int)

# 3. Hacer el merge
df_comuna = df_comuna.merge(
    df_pol[['Distrito', 'polarizacion_distrito']],
    on='Distrito',
    how='left'
)

# 4. Verificar que no haya quedado ninguna comuna sin polarización
print("=== VERIFICACIÓN ===")
print(f"Comunas totales: {len(df_comuna)}")
print(f"Comunas con polarización nula: {df_comuna['polarizacion_distrito'].isnull().sum()}")

print("\n=== ESTADÍSTICAS DE POLARIZACIÓN ASIGNADA A COMUNAS ===")
print(df_comuna['polarizacion_distrito'].describe().round(3).to_string())

# 5. Ver primeras filas para confirmar
print("\n=== PRIMERAS 5 FILAS ===")
print(df_comuna[['Comuna', 'Distrito', 'polarizacion_distrito']].head().to_string())

# 6. Guardar el dataset final
df_comuna.to_csv(r"C:\Users\Martín\Desktop\POLARIZACOIN\resultados\dataset_comuna_completo.csv", index=False, encoding='utf-8-sig')
print("\nGuardado: resultados/dataset_comuna_completo.csv")

=== VERIFICACIÓN ===
Comunas totales: 346
Comunas con polarización nula: 0

=== ESTADÍSTICAS DE POLARIZACIÓN ASIGNADA A COMUNAS ===
count    346.000
mean       1.357
std        0.407
min        0.000
25%        1.058
50%        1.407
75%        1.631
max        1.927

=== PRIMERAS 5 FILAS ===
        Comuna  Distrito  polarizacion_distrito
0  ANTOFAGASTA         3               1.260532
1       CALAMA         3               1.260532
2  MARIA ELENA         3               1.260532
3   MEJILLONES         3               1.260532
4      OLLAGUE         3               1.260532

Guardado: resultados/dataset_comuna_completo.csv


In [12]:
import pandas as pd

# 1. Contar cuántos diputados (de los 151 cruzados) quedaron en cada distrito
conteo_distritos = df_con_distrito.groupby('Distrito').size().reset_index(name='n_diputados_cruzados')

# 2. Mostrar la tabla
print("=== DIPUTADOS CRUZADOS POR DISTRITO ===")
print(conteo_distritos.to_string(index=False))

print("\n=== DISTRITOS CON POCOS DIPUTADOS (Sospechosos) ===")
sospechosos = conteo_distritos[conteo_distritos['n_diputados_cruzados'] < 3]
if len(sospechosos) > 0:
    print(sospechosos.to_string(index=False))
    print("Estos distritos probablemente tengan una polarización mal calculada (cercana a 0).")
else:
    print("Todos los distritos tienen 3 o más diputados cruzados.")

# 3. Ver cuáles son los distritos con polarización 0 en el dataset final
df_comuna = pd.read_csv(r"C:\Users\Martín\Desktop\POLARIZACOIN\resultados\dataset_comuna_completo.csv")
pol_cero = df_comuna[df_comuna['polarizacion_distrito'] == 0][['Comuna', 'Distrito', 'polarizacion_distrito']].drop_duplicates()
print("\n=== DISTRITOS CON POLARIZACIÓN 0 ASIGNADA ===")
print(pol_cero.to_string(index=False))

=== DIPUTADOS CRUZADOS POR DISTRITO ===
 Distrito  n_diputados_cruzados
        1                     3
        2                     3
        3                     5
        4                     5
        5                     6
        6                     8
        7                     7
        8                     8
        9                     7
       10                     8
       11                     6
       12                     7
       13                     5
       14                     6
       15                     5
       16                     4
       17                     7
       18                     3
       19                     5
       20                     8
       21                     5
       22                     4
       23                     7
       24                     5
       25                     4
       26                     4
       27                     3
       28                     3

=== DISTRITOS CON POCOS DIPUTAD

In [13]:
import pandas as pd
import numpy as np

# 1. Buscar los 6 nombres en SERVEL
nombres_faltantes = [
    'MANOUCHEHRI', 'SAEZ QUIROZ', 'SANCHEZ OSSA', 
    'VELOSO AVILA', 'CELEDON FERNANDEZ', 'BARRIOS OTEIZA'
]

print("=== BÚSQUEDA MANUAL DE LOS 6 DIPUTADOS EN SERVEL ===")
for nombre in nombres_faltantes:
    coincidencia = df_serval_candidatos[
        df_serval_candidatos['Candidato'].str.upper().str.contains(nombre, na=False)
    ][['Candidato', 'Distrito']].drop_duplicates()
    print(f"\nBuscando '{nombre}':")
    print(coincidencia.to_string(index=False))

# 2. PARCHE MANUAL: Asignar distritos según la búsqueda (ajustar si es necesario)
# (Asumiendo que los resultados de la búsqueda mostrarán el distrito correcto)
parche_manual = {
    'Daniel Manouchehri Lobos': 5, # (Reemplazar con el distrito real de la búsqueda)
    'Jaime Sáez Quiroz': 28,       # (Reemplazar con el distrito real)
    'Luis Sánchez Ossa': 6,        # (Reemplazar con el distrito real)
    'Consuelo Veloso Ávila': 28,   # (Reemplazar con el distrito real)
    'Roberto Celedón Fernández': 28, # (Reemplazar con el distrito real)
    'Arturo Barrios Oteíza': 1     # (Reemplazar con el distrito real)
}

=== BÚSQUEDA MANUAL DE LOS 6 DIPUTADOS EN SERVEL ===

Buscando 'MANOUCHEHRI':
                               Candidato  Distrito
DANIEL MANOUCHEHRI MOGHADAM KASHAN LOBOS       5.0

Buscando 'SAEZ QUIROZ':
                 Candidato  Distrito
JAIME SALVADOR SAEZ QUIROZ      26.0

Buscando 'SANCHEZ OSSA':
                 Candidato  Distrito
LUIS FERNANDO SANCHEZ OSSA       7.0

Buscando 'VELOSO AVILA':
                           Candidato  Distrito
CONSUELO DE LOS ANGELES VELOSO AVILA      18.0

Buscando 'CELEDON FERNANDEZ':
Empty DataFrame
Columns: [Candidato, Distrito]
Index: []

Buscando 'BARRIOS OTEIZA':
Empty DataFrame
Columns: [Candidato, Distrito]
Index: []


In [14]:
import pandas as pd

# 1. Aplicar el parche manual a df_p6 (que tiene los 157 diputados)
parche_manual = {
    'Daniel Manouchehri Lobos': 5,
    'Jaime Sáez Quiroz': 26,
    'Luis Sánchez Ossa': 7,
    'Consuelo Veloso Ávila': 18
}

for nombre, distrito in parche_manual.items():
    # Buscar la fila y asignar el distrito
    df_p6.loc[df_p6['Name'] == nombre, 'Distrito'] = distrito

# Eliminar a los 2 que no encontramos (reemplazantes) para que no generen nulos
df_final_diputados = df_p6.dropna(subset=['Distrito']).copy()
df_final_diputados['Distrito'] = df_final_diputados['Distrito'].astype(int)

# 2. Recalcular la polarización por distrito
# Agrupar por distrito y bloque para obtener la posición media
posicion_por_distrito = df_final_diputados.groupby(['Distrito', 'Bloque'])['wnom_ideology'].mean().reset_index()

# Calcular los extremos (min y max) por distrito
extremos = posicion_por_distrito.groupby('Distrito')['wnom_ideology'].agg(['min', 'max']).reset_index()
extremos = extremos.rename(columns={'min': 'ideologia_min', 'max': 'ideologia_max'})
extremos['polarizacion_distrito'] = extremos['ideologia_max'] - extremos['ideologia_min']

print("=== VERIFICACIÓN DE POLARIZACIÓN ===")
print(f"Distritos calculados: {len(extremos)}")
print(f"Polarización mínima: {extremos['polarizacion_distrito'].min():.3f}")
print(f"Polarización máxima: {extremos['polarizacion_distrito'].max():.3f}")

# Ver el Distrito 28 que antes daba 0
print("\nDistrito 28:")
print(extremos[extremos['Distrito'] == 28].to_string(index=False))

# 3. Actualizar el dataset de comunas
ruta_completo = r"C:\Users\Martín\Desktop\POLARIZACOIN\resultados\dataset_comuna_completo.csv"
df_comuna = pd.read_csv(ruta_completo)

# Eliminar la columna de polarización vieja (la que tenía ceros)
df_comuna = df_comuna.drop(columns=['polarizacion_distrito'], errors='ignore')

# Hacer el merge corregido
df_comuna['Distrito'] = df_comuna['Distrito'].astype(int)
df_comuna = df_comuna.merge(
    extremos[['Distrito', 'polarizacion_distrito']],
    on='Distrito',
    how='left'
)

print("\n=== DATASET COMUNAL ACTUALIZADO ===")
print(f"Nulos en polarización: {df_comuna['polarizacion_distrito'].isnull().sum()}")
print(f"Polarización mínima en comunas: {df_comuna['polarizacion_distrito'].min():.3f}")

# Guardar el dataset definitivo
df_comuna.to_csv(ruta_completo, index=False, encoding='utf-8-sig')
print("\nGuardado: resultados/dataset_comuna_completo.csv")

=== VERIFICACIÓN DE POLARIZACIÓN ===
Distritos calculados: 28
Polarización mínima: 0.000
Polarización máxima: 1.927

Distrito 28:
 Distrito  ideologia_min  ideologia_max  polarizacion_distrito
       28      -0.272422      -0.272422                    0.0

=== DATASET COMUNAL ACTUALIZADO ===
Nulos en polarización: 0
Polarización mínima en comunas: 0.000

Guardado: resultados/dataset_comuna_completo.csv


In [15]:
import pandas as pd

# Ver los diputados que sobrevivieron en el Distrito 28
print("=== DIPUTADOS EN EL DISTRITO 28 ===")
print(df_final_diputados[df_final_diputados['Distrito'] == 28][['Name', 'party_1', 'Bloque', 'wnom_ideology']].to_string(index=False))

=== DIPUTADOS EN EL DISTRITO 28 ===
                     Name party_1                    Bloque  wnom_ideology
   Carlos Bianchi Chelech     IND CANDIDATURA INDEPENDIENTE      -0.513954
Christian Matheson Villán     IND CANDIDATURA INDEPENDIENTE       0.548137
 Javiera Morales Alvarado     IND CANDIDATURA INDEPENDIENTE      -0.851449


In [16]:
import pandas as pd

# 1. Recalcular extremos usando la posición INDIVIDUAL de cada diputado
# (No promediamos por bloque, tomamos el min y max de los diputados)
extremos_corregidos = df_final_diputados.groupby('Distrito')['wnom_ideology'].agg(['min', 'max']).reset_index()
extremos_corregidos = extremos_corregidos.rename(columns={'min': 'ideologia_min', 'max': 'ideologia_max'})
extremos_corregidos['polarizacion_distrito'] = extremos_corregidos['ideologia_max'] - extremos_corregidos['ideologia_min']

print("=== VERIFICACIÓN DE POLARIZACIÓN CORREGIDA ===")
print(f"Polarización mínima: {extremos_corregidos['polarizacion_distrito'].min():.3f}")
print(f"Polarización máxima: {extremos_corregidos['polarizacion_distrito'].max():.3f}")

print("\nDistrito 28 (Magallanes):")
print(extremos_corregidos[extremos_corregidos['Distrito'] == 28].to_string(index=False))

# 2. Actualizar el dataset de comunas
ruta_completo = r"C:\Users\Martín\Desktop\POLARIZACOIN\resultados\dataset_comuna_completo.csv"
df_comuna = pd.read_csv(ruta_completo)

# Eliminar la columna vieja
df_comuna = df_comuna.drop(columns=['polarizacion_distrito'], errors='ignore')

# Hacer el merge con los datos corregidos
df_comuna = df_comuna.merge(
    extremos_corregidos[['Distrito', 'polarizacion_distrito']],
    on='Distrito',
    how='left'
)

print("\n=== DATASET COMUNAL FINAL ===")
print(f"Nulos en polarización: {df_comuna['polarizacion_distrito'].isnull().sum()}")
print(f"Polarización mínima: {df_comuna['polarizacion_distrito'].min():.3f}")
print(f"Polarización máxima: {df_comuna['polarizacion_distrito'].max():.3f}")

# Guardar
df_comuna.to_csv(ruta_completo, index=False, encoding='utf-8-sig')
print("\nGuardado: resultados/dataset_comuna_completo.csv")

=== VERIFICACIÓN DE POLARIZACIÓN CORREGIDA ===
Polarización mínima: 0.851
Polarización máxima: 1.958

Distrito 28 (Magallanes):
 Distrito  ideologia_min  ideologia_max  polarizacion_distrito
       28      -0.851449       0.548137               1.399586

=== DATASET COMUNAL FINAL ===
Nulos en polarización: 0
Polarización mínima: 0.851
Polarización máxima: 1.958

Guardado: resultados/dataset_comuna_completo.csv


In [ ]:
import pandas as pd
import numpy as np
from scipy.stats import spearmanr
from scipy import stats

# 1. Cargar el dataset final
ruta_completo = r"C:\Users\Martín\Desktop\POLARIZACOIN\resultados\dataset_comuna_completo.csv"
df = pd.read_csv(ruta_completo)

# 2. Correlaciones de Spearman
variables = [
    'edad_promedio', 'pct_jovenes', 'promedio_escolaridad', 'pct_superior',
    'pct_rural', 'tasa_desocupacion', 'pct_evangelicos_limpio', 
    'pct_sin_religion_limpio', 'tasa_pobreza_ingresos_2024', 'Laakso_Taagepera'
]

print("=== CORRELACIÓN CON POLARIZACIÓN LEGISLATIVA (Spearman) ===")
print(f"{'Variable':<30} {'r':>8} {'p':>10} {'N':>5}")
print("-" * 55)
for var in variables:
    subset = df[['polarizacion_distrito', var]].dropna()
    r, p = spearmanr(subset['polarizacion_distrito'], subset[var])
    sig = "***" if p<0.001 else "**" if p<0.01 else "*" if p<0.05 else ""
    print(f"{var:<30} {r:>8.3f} {p:>10.4f} {len(subset):>5} {sig}")

# 3. Regresión múltiple estandarizada
vars_x = ['edad_promedio', 'promedio_escolaridad', 'pct_rural', 'tasa_desocupacion']
var_y = 'polarizacion_distrito'

df_modelo = df[vars_x + [var_y]].dropna().copy()
df_z = (df_modelo - df_modelo.mean()) / df_modelo.std()

X = np.column_stack([np.ones(len(df_z))] + [df_z[v].values for v in vars_x])
y = df_z[var_y].values

coefs = np.linalg.lstsq(X, y, rcond=None)[0]
residuos = y - (X @ coefs)
r2 = 1 - (np.sum(residuos ** 2) / np.sum((y - y.mean()) ** 2))

n = len(y)
k = X.shape[1] - 1
se = np.sqrt(np.diag((np.sum(residuos ** 2) / (n - k - 1)) * np.linalg.inv(X.T @ X)))
t_stats = coefs / se
p_values = 2 * (1 - stats.t.cdf(np.abs(t_stats), df=n - k - 1))

print(f"\n=== REGRESIÓN MÚLTIPLE PARA POLARIZACIÓN ===")
print(f"R² = {r2:.3f}  (Comparación: Fragmentación fue 0.109)")
print(f"\n{'Variable':<25} {'Coef':>8} {'SE':>8} {'p':>10}")
print("-" * 55)
for i, var in enumerate(vars_x, 1):
    sig = "***" if p_values[i]<0.001 else "**" if p_values[i]<0.01 else "*" if p_values[i]<0.05 else ""
    print(f"{var:<25} {coefs[i]:>8.3f} {se[i]:>8.3f} {p_values[i]:>10.4f} {sig}")